<a href="https://colab.research.google.com/github/adeleke1508/flyrank_week_one_ml_project_1/blob/main/work/notebooks/w01_research_question.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/adeleke1508/flyrank_week_one_ml_project_1/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

Lane: Structured Content Archetype Clustering

I'm choosing this lane because it stretches me into unsupervised learning, which I haven't worked with yet — my background so far (Walmart sales prediction, Carvana segmentation, NLI classification) has all been supervised, where there's a clear right answer to check against. Clustering forces a different kind of thinking: grouping pages by behavior without labels, then validating that the groups are real and not just noise. It's also a natural fit for FlyRank's problem — with hundreds of thousands of content items, a human reviewer can't look at each one individually, but if pages cluster into recognizable types (e.g. high performers, quietly declining, high-effort-low-return), that gives a reviewer a fast mental map of the whole content inventory and a starting point for triage. I'll confirm this choice against real signal-quality checks (cluster stability, PCA variance explained) in the next couple weeks before locking it in.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

What decision does your work improve?
This work improves the decision of which pages to review first, and what action to take, out of a content inventory too large to review manually.

Who acts on it?
The person acting on it is a content reviewer or SEO strategist with limited time.

What does a wrong recommendation cost?
A wrong recommendation doesn't cause direct financial harm — it costs reviewer hours spent on the wrong pages, and risks a truly declining page being hidden inside a cluster labeled "healthy." The bigger long-term risk is trust: if clusters are unstable or don't hold up to scrutiny, reviewers will stop relying on the tool altogether — so validating cluster stability isn't optional, it's central to whether this lane produces anything usable.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

Running the starter dataset below through a quick clustering pass shows there's real structure to work with. Out of the filtered dataset, 30000 rows had enough usable numeric signal (impressions, clicks, position, engagement, etc.) to cluster on. A 2-dimensional PCA projection captured 44.35% of the variance in that data — meaning most of what separates one page from another can be summarized along just a couple of underlying dimensions, rather than being pure noise. A first-pass KMeans with 5 clusters produced a silhouette score of 0.279, indicating a modest but real separation between groups. Even at this rough first pass, that's enough signal to justify spending the next 7 weeks refining feature selection, testing cluster counts, and validating whether these groups hold up to human inspection.

In [4]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    # find the repo root from wherever this kernel started
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"
print("Starter data found. You're ready.")

Working dir: /content/flyrank-ml-internship-starter
Starter data found. You're ready.


In [7]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

df = pd.read_csv('data/raw/content_refresh_anonymized.csv')

df = df[(df['impressions_90d'] > 0) & (df['content_age_days'] >= 90) ]
df = df.drop_duplicates(subset='content_id')

numeric_cols = ['impressions_90d', 'clicks_90d', 'sessions_90d',
                 'content_age_days', 'word_count', 'avg_position',
                 'ctr', 'engagement_rate', 'scroll_rate']
cols = [c for c in numeric_cols if c in df.columns]

X = df[cols].replace([np.inf, -np.inf], np.nan).fillna(0)
print(f'Rows available for clustering:{len(X)}')
print(f'Numeric features available: {len(numeric_cols)} -> {numeric_cols}')

X_scaled = StandardScaler().fit_transform(X)

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)
print(f'Variance explained by first two components: {pca.explained_variance_ratio_.sum():.2%}')

kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)
labels = kmeans.fit_predict(X_scaled)
sil_score = silhouette_score(X_scaled, labels)

print(f'Silhouette score (5 clusters): {sil_score:.3f}')
print(f'Cluster size: {np.bincount(labels)}')

Rows available for clustering:30000
Numeric features available: 9 -> ['impressions_90d', 'clicks_90d', 'sessions_90d', 'content_age_days', 'word_count', 'avg_position', 'ctr', 'engagement_rate', 'scroll_rate']
Variance explained by first two components: 44.35%
Silhouette score (5 clusters): 0.279
Cluster size: [  611  3394 13816 11879   300]


## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

My clustering work will produce observed findings (e.g. "cluster X shows a consistent pattern of high impressions with low CTR") and directional signals (e.g. "pages in cluster Y are disproportionately old and unrefreshed, which may explain their decline"). These are meant for decision support — helping a reviewer prioritize which group of pages to look at first, not telling them the definitive answer.

This work will never claim causal proof — I can't say a refresh caused a recovery without running an actual experiment, which this data doesn't support. It also won't make any claims about how Google's algorithm works or what it "rewards" — I'm only working with observed clicks, impressions, and position, not the search engine's internal logic. Any language suggesting otherwise would be overstating what correlational, non-experimental data can prove.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.